# Training Linear Regression and XGBoost models on data

## Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8")

### Load Data

In [47]:
train_transformed = pd.read_csv('../../data/preprocessed/train_preprocessed.csv')
x_train_transformed = train_transformed.drop("diabetes", axis=1)
y_train_transformed = train_transformed["diabetes"]

val = pd.read_csv('../../data/preprocessed/validation_preprocessed.csv')
x_val = val.drop("diabetes", axis=1)
y_val = val["diabetes"]

print(f'x_train_transformed: {train_transformed.shape}')
train_transformed.head()

x_train_transformed: (67139, 21)


,smoking_history_current,smoking_history_ever,smoking_history_former,smoking_history_never,smoking_history_not current,age,bmi,HbA1c_level,blood_glucose_level,glucose_hba1c_interaction,...,age_bmi_interaction,bmi_hba1c_interaction,age_glucose_interaction,gender,hypertension,heart_disease,high_hba1c_flag,senior_flag,cardio_risk_flag,diabetes
0,0.0,0.0,0.0,0.0,0.0,0.612112,-0.637209,0.000000,-0.677966,-0.459103,...,-0.056612,-0.232854,-0.081827,0,0,0,0,0,0,0
1,0.0,0.0,1.0,0.0,0.0,0.737237,0.818605,0.500000,0.000000,0.411609,...,0.658534,1.018393,0.557564,0,0,0,0,0,0,0
2,0.0,0.0,1.0,0.0,0.0,0.399399,-0.229457,0.000000,0.254237,0.382586,...,-0.339001,0.014118,-0.070409,1,0,0,0,0,0,0
3,0.0,1.0,0.0,0.0,0.0,0.874875,1.485271,0.500000,0.254237,0.668865,...,1.258590,1.470922,1.050428,0,0,0,0,1,0,0
4,0.0,0.0,0.0,1.0,0.0,0.687187,-0.395349,-1.642857,1.016949,-0.142480,...,0.148131,-1.008759,1.078972,1,0,0,0,0,0,0


In [48]:
train_ADASYN = pd.read_csv('../../data/imbalance_resolve/ADASYN.csv')
x_train_ADASYN = train_ADASYN.drop("diabetes_target", axis=1)
y_train_ADASYN = train_ADASYN["diabetes_target"]
print(f'train_ADASYN: {train_ADASYN.shape}')

train_smote_enn = pd.read_csv('../../data/imbalance_resolve/train_smote_enn.csv')
x_train_smote_enn = train_smote_enn.drop("diabetes_target", axis=1)
y_train_smote_enn = train_smote_enn["diabetes_target"]
print(f'train_smote_enn: {train_smote_enn.shape}')

train_smote_tomek = pd.read_csv('../../data/imbalance_resolve/train_smote_tomek.csv')
x_train_smote_tomek= train_smote_tomek.drop("diabetes_target", axis=1)
y_train_smote_tomek = train_smote_tomek["diabetes_target"]
print(f'train_smote_tomek: {train_smote_tomek.shape}')

train_smote = pd.read_csv('../../data/imbalance_resolve/train_smote.csv')
x_train_smote = train_smote.drop("diabetes_target", axis=1)
y_train_smote = train_smote["diabetes_target"]
print(f'train_smote: {train_smote.shape}')

train_ADASYN: (122666, 16)
train_smote_enn: (112303, 16)
train_smote_tomek: (121658, 16)
train_smote: (112303, 16)


### Evaluation Metrics
- Accuracy
- **Precision**: When I say diabetic, how often am I right?, TP / (TP + FP)
- **Recall**: How many diabetics did I catch?, TP / (TP + FN) :: **most important**
- **F1-score**
- ROC-AUC
- **PR-AUC**
- **MCC**

`it is more important to us to say that a patient has diabetes when he has, and missing that would be bad that is why recall is most important here`

In [27]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,  # PR-AUC
    matthews_corrcoef
)

#### Helper function for model evaluation

In [35]:
def evaluate_model(model, X, y, threshold=0.5):
    y_proba = model.predict_proba(X)[:, 1]

    y_pred = (y_proba >= threshold).astype(int)

    results = {
        "Accuracy": accuracy_score(y, y_pred),
        "Precision": precision_score(y, y_pred, zero_division=0),
        "Recall": recall_score(y, y_pred),
        "F1-score": f1_score(y, y_pred),
        "ROC-AUC": roc_auc_score(y, y_proba),
        "PR-AUC": average_precision_score(y, y_proba),
        "MCC": matthews_corrcoef(y, y_pred)
    }    

    return results

### Logistic Regression

I will first try logistic regression using different train data and select the best data to tune the hyperparameters:
- x_train_transformed
- x_train_transformed (with weights)
- x_train_ADASYN
- x_train_smote_enn
- x_train_smote_tomek
- x_train_smote

#### import model

In [16]:
from sklearn.linear_model import LogisticRegression

#### x_train_transformed

In [49]:
model_LR_transformed = LogisticRegression(max_iter=1000, random_state=42)

model_LR_transformed.fit(x_train_transformed, y_train_transformed)

results = evaluate_model(model_LR_transformed, x_val, y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.9607
Precision: 0.8716
Recall: 0.6509
F1-score: 0.7453
ROC-AUC: 0.9622
PR-AUC: 0.8298
MCC: 0.7335


#### x_train_transformed (with weights)

In [50]:
model_LR_transformed_w = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")

model_LR_transformed_w.fit(x_train_transformed, y_train_transformed)

results = evaluate_model(model_LR_transformed_w, x_val, y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.8864
Precision: 0.4308
Recall: 0.8852
F1-score: 0.5795
ROC-AUC: 0.9628
PR-AUC: 0.8251
MCC: 0.5682


#### x_train_ADASYN

In [ ]:
model_LR_ADASYN = LogisticRegression(max_iter=1000, random_state=42)

model_LR_ADASYN.fit(x_train_ADASYN, y_train_ADASYN)

results = evaluate_model(model_LR_ADASYN, x_val[x_train_ADASYN.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.7990
Precision: 0.2996
Recall: 0.9520
F1-score: 0.4558
ROC-AUC: 0.9586
PR-AUC: 0.8035
MCC: 0.4650


#### x_train_smote_enn

In [51]:
model_LR_smote_enn = LogisticRegression(max_iter=1000, random_state=42)

model_LR_smote_enn.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_LR_smote_enn, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.8508
Precision: 0.3647
Recall: 0.9269
F1-score: 0.5234
ROC-AUC: 0.9602
PR-AUC: 0.8098
MCC: 0.5239


#### x_train_smote_tomek

In [52]:
model_LR_smote_tomek = LogisticRegression(max_iter=1000, random_state=42)

model_LR_smote_tomek.fit(x_train_smote_tomek, y_train_smote_tomek)

results = evaluate_model(model_LR_smote_tomek, x_val[x_train_smote_tomek.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.8649
Precision: 0.3870
Recall: 0.9033
F1-score: 0.5419
ROC-AUC: 0.9594
PR-AUC: 0.8079
MCC: 0.5363


#### x_train_smote

In [53]:
model_LR_smote = LogisticRegression(max_iter=1000, random_state=42)

model_LR_smote.fit(x_train_smote, y_train_smote)

results = evaluate_model(model_LR_smote, x_val[x_train_smote.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Accuracy: 0.8508
Precision: 0.3647
Recall: 0.9269
F1-score: 0.5234
ROC-AUC: 0.9602
PR-AUC: 0.8098
MCC: 0.5239


#### Results

| Method                     | Accuracy | Precision | Recall | F1-score | ROC-AUC | PR-AUC | MCC   |
|---------------------------|----------|-----------|--------|----------|---------|--------|-------|
| x_train_transformed       | `0.9607`   | `0.8716`    | 0.6509 | `0.7453`   | 0.9622  | `0.8298` | `0.7335` |
| x_train_transformed (w)   | 0.8864   | 0.4308    | 0.8852 | 0.5795   | `0.9628`  | 0.8251 | 0.5682 |
| x_train_ADASYN            | 0.7990   | 0.2996    | `0.9520` | 0.4558   | 0.9586  | 0.8035 | 0.4650 |
| x_train_smote_enn         | 0.8508   | 0.3647    | 0.9269 | 0.5234   | 0.9602  | 0.8098 | 0.5239 |
| x_train_smote_tomek       | 0.8649   | 0.3870    | 0.9033 | 0.5419   | 0.9594  | 0.8079 | 0.5363 |
| x_train_smote             | 0.8508   | 0.3647    | 0.9269 | 0.5234   | 0.9602  | 0.8098 | 0.5239 |

we will continue with the x_train_smote_enn to tune the hyperparemeters

#### Tuning hyperparameters using x_train_smote_enn

##### E1

In [69]:
model_LR_smote_enn = LogisticRegression(
    penalty='l2',
    C=1.0,
    solver='liblinear',
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

model_LR_smote_enn.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_LR_smote_enn, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

c:\Users\ahmed\AppData\Local\pypoetry\Cache\virtualenvs\diabetes-prediction-7fFvrh6q-py3.13\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Accuracy: 0.8536
Precision: 0.3693
Recall: 0.9261
F1-score: 0.5280
ROC-AUC: 0.9603
PR-AUC: 0.8100
MCC: 0.5281


##### E2

In [70]:
model_LR_smote_enn = LogisticRegression(
    penalty='l2',
    C=0.1,
    solver='liblinear',
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

model_LR_smote_enn.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_LR_smote_enn, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

c:\Users\ahmed\AppData\Local\pypoetry\Cache\virtualenvs\diabetes-prediction-7fFvrh6q-py3.13\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Accuracy: 0.8553
Precision: 0.3718
Recall: 0.9237
F1-score: 0.5302
ROC-AUC: 0.9605
PR-AUC: 0.8109
MCC: 0.5296


##### E3

In [ ]:
model_LR_smote_enn = LogisticRegression(
    penalty='l2',
    C=0.01,
    solver='liblinear',
    class_weight='balanced',    
    max_iter=1000,
    random_state=42
)

model_LR_smote_enn.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_LR_smote_enn, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

c:\Users\ahmed\AppData\Local\pypoetry\Cache\virtualenvs\diabetes-prediction-7fFvrh6q-py3.13\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Accuracy: 0.8631
Precision: 0.3849
Recall: 0.9167
F1-score: 0.5422
ROC-AUC: 0.9612
PR-AUC: 0.8149
MCC: 0.5394


##### E4

In [78]:
model_LR_smote_enn = LogisticRegression(
    penalty='l2',
    C=0.001,
    solver='liblinear',
    class_weight='balanced',    
    max_iter=1000,
    random_state=42
)

model_LR_smote_enn.fit(x_train_smote_enn, y_train_smote_enn)

results = evaluate_model(model_LR_smote_enn, x_val[x_train_smote_enn.columns], y_val)

for k, v in results.items():
    print(f"{k}: {v:.4f}")

c:\Users\ahmed\AppData\Local\pypoetry\Cache\virtualenvs\diabetes-prediction-7fFvrh6q-py3.13\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Accuracy: 0.8546
Precision: 0.3703
Recall: 0.9206
F1-score: 0.5282
ROC-AUC: 0.9605
PR-AUC: 0.8143
MCC: 0.5271


#### Results

| C value | Accuracy | Precision | Recall | F1-score | ROC-AUC | PR-AUC | MCC   |
|--------|----------|-----------|--------|----------|---------|--------|-------|
| C=1.0  | 0.8536   | 0.3693    | 0.9261 | 0.5280   | 0.9603  | 0.8100 | 0.5281 |
| C=0.1  | 0.8553   | 0.3718    | 0.9237 | 0.5302   | 0.9605  | 0.8109 | 0.5296 |
| C=0.01 | 0.8631   | 0.3849    | 0.9167 | 0.5422   | 0.9612  | 0.8149 | 0.5394 |
| C=0.001 | 0.8546   | 0.3703    | 0.9206 | 0.5282   | 0.9605  | 0.8143 | 0.5271 |


`choose C=0.01`: it gives good trade off between Accuracy, Precision, Recall